In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json

In [ ]:
expts = {'XLI2':'Schisto',
         'XCE1':'CoV',
         'JYH3':'BoNT/A'}
expt_order = ['XLI2','JYH3','XCE1']
alg_order = ['individual','meta','scoper','mmseqs']

scores = []
for e in expt_order:
    for a in alg_order:
        sc = pd.read_csv(f'../fig3_meta-clustering_validation/{e}_{a}_scores_extra.csv',index_col=0)
        sc['expt'] = e
        sc['alg'] = a

        if a == 'scoper':
            sc['max_dist'] = 1 - sc['max_dist'] #fix mistake
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        elif a=='mmseqs':
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score_v3']/(-50)
        elif a=='individual':
            sc['max_dist'] = sc['cut']
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        else:
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        
        scores.append(sc)

scores = pd.concat(scores,axis=0).reset_index(drop=True)
scores

In [ ]:
best_settings = json.load(open('../fig3_meta-clustering_validation/best_params.json','r'))
best_settings = {tuple(k.split('-')):best_settings[k] for k in best_settings}
best_settings

In [ ]:
for k in best_settings:
    row = scores[(scores['factor']==1)&(scores['name']==best_settings[k])&(scores['expt']==k[1])]
    print(k, row['size'].values)

In [ ]:
colorblind = sns.color_palette("colorblind")
tab10 = plt.get_cmap('tab10')
letters = 'ABCD'

color_id = {
    'individual':9,
    'meta':0,
    'scoper':1,
    'mmseqs':2
}

count_names = {
    'N_fam': 'Families',
    'N_enrich': 'Enriched Fam.',
    'N_enrich_groups': 'Enriched Grp.',
}

tick_locs = []
ticks = []
f, (ax, ax2) = plt.subplots(2, 1, sharex=True)
for e,expt in enumerate(expt_order):
    for a,alg in enumerate(alg_order):
        base_color = np.array(colorblind[color_id[alg]])
        print(expt, alg, scores.loc[(scores['name']==best_settings[(alg,expt)])&(scores['factor']==1)&(scores['expt']==expt),'N_fam'].values[0])
        for v,val in enumerate(['N_enrich','N_enrich_groups']):
            x = 5*e + a
            y = scores.loc[(scores['name']==best_settings[(alg,expt)])&(scores['factor']==1)&(scores['expt']==expt),val]

            if (e==0) and (a==3):
                label = count_names[val]
                ax.bar(x,y,color=base_color+((1-base_color)*(1-((v+1)/2))),width=1,label=label)
            elif e==0:
                ax.bar(x,y,color=base_color+((1-base_color)*(1-((v+1)/2))),width=1,label=' ')
            else:
                ax.bar(x,y,color=base_color+((1-base_color)*(1-((v+1)/2))),width=1)
            ax2.bar(x,y,color=base_color+((1-base_color)*(1-((v+1)/2))),width=1)
                
    tick_locs.append(x-1.5)
    ticks.append(expts[expt])

d = .015  # how big to make the diagonal lines in axes coordinates
# arguments to pass to plot, just so we don't keep repeating them
kwargs = dict(transform=ax.transAxes, color='k', clip_on=False)
ax.plot((-d, +d), (-d, +d), **kwargs)        # top-left diagonal
ax.plot((1 - d, 1 + d), (-d, +d), **kwargs)  # top-right diagonal

kwargs.update(transform=ax2.transAxes)  # switch to the bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # bottom-left diagonal
ax2.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)  # bottom-right diagonal

ax.set_ylim(350, 750)   # Top plot
ax2.set_ylim(100, 250) # Bottom plot
    
ax2.set_xticks(tick_locs,ticks,fontsize=16,fontweight='bold')
for l, label in enumerate(ax2.get_xticklabels()):
    label.set_color(tab10(l+3))

ax.set_yticks([400,500,600,700],[400,500,600,700],fontsize=16)
ax2.set_yticks([100,150,200],[100,150,200],fontsize=16)

ax.spines['bottom'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax.xaxis.tick_top()
ax.tick_params(labeltop=False)
ax2.xaxis.tick_bottom()

ax.legend(ncols=4,title='NanoMAP  NanoMAP'+'\n    (ind)       (meta)    SCOPer   MMseqs2'+' '*25,loc=[0.17,1.02])

plt.savefig('figS16.png',dpi=300,bbox_inches='tight')